# Week 2 · Day 3 — Custom Dataset Pipeline
**Dataset, DataLoader & Transforms on a Kaggle Dataset**

| | |
|---|---|
| **Runtime** | GPU (T4 — free tier) |
| **Estimated time** | ~60–90 min |
| **Topics** | ImageFolder · Custom Dataset · Transforms · Data analysis · Transfer learning on custom data |

---
### ⚡ Before you start
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Upload the **Intel Image Classification** dataset to your Google Drive at:
   `MyDrive/datasets/intel-image-classification/`
3. Run cells top-to-bottom

## 0 · Setup & GPU Check

In [ ]:
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {DEVICE}')

In [ ]:
!pip install -q wandb
print('✅ Packages ready')

In [ ]:
# Mount Google Drive — dataset + results persistence
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Paths on Google Drive ──────────────────────────────────────
DRIVE_DATA_DIR    = '/content/drive/MyDrive/datasets/intel-image-classification'
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/ComputerVision/week2_classification/results'
DRIVE_MODEL_DIR   = '/content/drive/MyDrive/ComputerVision/week2_classification/models'

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

# Verify dataset exists
assert os.path.exists(DRIVE_DATA_DIR), (
    f'❌ Dataset not found at {DRIVE_DATA_DIR}\n'
    f'   Upload Intel Image Classification to your Drive at:\n'
    f'   MyDrive/datasets/intel-image-classification/'
)
print(f'Dataset  → {DRIVE_DATA_DIR}')
print(f'Results  → {DRIVE_RESULTS_DIR}')
print(f'Models   → {DRIVE_MODEL_DIR}')
print('✅ Google Drive mounted & paths verified')

## 1 · Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
import torchvision.models as models
from torchvision.models import ResNet18_Weights, ResNet50_Weights, EfficientNet_B0_Weights
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110
import numpy as np
from pathlib import Path
from collections import Counter
import time
import json

torch.manual_seed(42)
np.random.seed(42)
print('✅ Imports done')

## 2 · Configuration

In [ ]:
# ─── CONFIGURATION ────────────────────────────────────────────
BACKBONE    = 'resnet18'     # 'resnet18' | 'resnet50' | 'efficientnet_b0'
EPOCHS      = 10             # 5 for quick test, 15-20 for serious run
BATCH_SIZE  = 32             # 32 is good for 150px images on T4
LR_HEAD     = 1e-3           # learning rate for classifier head
LR_BACKBONE = 1e-5           # learning rate for backbone (fine-tuning)
WEIGHT_DECAY= 1e-4
IMG_SIZE    = 224            # resize to this (pretrained models expect ~224)
VAL_SPLIT   = 0.15           # fraction of training data for validation
USE_WANDB   = False          # set True and run wandb.login() below
# ──────────────────────────────────────────────────────────────

# ImageNet normalisation — required for pretrained backbones
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

print(f'Backbone  : {BACKBONE}')
print(f'Epochs    : {EPOCHS}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Img size  : {IMG_SIZE}px')
print(f'LR head / backbone : {LR_HEAD} / {LR_BACKBONE}')

In [ ]:
# Optional W&B login — skip if USE_WANDB = False
if USE_WANDB:
    import wandb
    wandb.login()
    print('✅ Logged into W&B')

---
## 3 · Dataset Exploration

**Always explore your data before training!**
- How many images per class?
- Are classes balanced?
- What do the images look like?
- What sizes are they?

In [ ]:
# ── Find and validate the dataset ──────────────────────────────
# Intel Image Classification has nested dirs: seg_train/seg_train/<class>/

train_root = os.path.join(DRIVE_DATA_DIR, 'seg_train', 'seg_train')
test_root  = os.path.join(DRIVE_DATA_DIR, 'seg_test', 'seg_test')

# Fallback: non-nested structure
if not os.path.exists(train_root):
    train_root = os.path.join(DRIVE_DATA_DIR, 'seg_train')
if not os.path.exists(test_root):
    test_root = os.path.join(DRIVE_DATA_DIR, 'seg_test')

assert os.path.exists(train_root), f'❌ Train dir not found: {train_root}'
assert os.path.exists(test_root),  f'❌ Test dir not found: {test_root}'

class_names = sorted([d for d in os.listdir(train_root)
                       if os.path.isdir(os.path.join(train_root, d))])
NUM_CLASSES = len(class_names)

print(f'Train dir: {train_root}')
print(f'Test dir:  {test_root}')
print(f'Classes ({NUM_CLASSES}): {class_names}')

In [ ]:
# ── Count images per class ────────────────────────────────────
print('\nDATASET ANALYSIS')
print('=' * 55)

train_counts = {}
test_counts  = {}
sample_sizes = []

for cls in class_names:
    train_cls = os.path.join(train_root, cls)
    test_cls  = os.path.join(test_root, cls)

    t_imgs = [f for f in os.listdir(train_cls) if not f.startswith('.')]
    e_imgs = [f for f in os.listdir(test_cls)  if not f.startswith('.')] if os.path.exists(test_cls) else []

    train_counts[cls] = len(t_imgs)
    test_counts[cls]  = len(e_imgs)

    # Sample a few image sizes
    for fname in t_imgs[:3]:
        try:
            img = Image.open(os.path.join(train_cls, fname))
            sample_sizes.append(img.size)
        except: pass

total_train = sum(train_counts.values())
total_test  = sum(test_counts.values())

print(f"{'Class':<15} {'Train':>8} {'Test':>8} {'% Train':>10}")
print('-' * 43)
for cls in class_names:
    pct = 100 * train_counts[cls] / total_train
    print(f'{cls:<15} {train_counts[cls]:>8} {test_counts[cls]:>8} {pct:>9.1f}%')
print('-' * 43)
print(f'{"TOTAL":<15} {total_train:>8} {total_test:>8}')

# Imbalance check
counts_list = list(train_counts.values())
ratio = max(counts_list) / min(counts_list)
print(f'\nImbalance ratio: {ratio:.2f}')
print('✅ Balanced' if ratio < 2 else '⚠️ Imbalanced — consider WeightedRandomSampler')

if sample_sizes:
    ws = [s[0] for s in sample_sizes]
    hs = [s[1] for s in sample_sizes]
    print(f'\nImage sizes: {min(ws)}×{min(hs)} to {max(ws)}×{max(hs)}')

In [ ]:
# ── Plot class distribution ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = plt.cm.Set3(np.linspace(0, 1, NUM_CLASSES))

axes[0].barh(class_names, [train_counts[c] for c in class_names], color=colors)
axes[0].set_xlabel('Number of Images')
axes[0].set_title('Training Set Distribution')
for i, cls in enumerate(class_names):
    axes[0].text(train_counts[cls]+10, i, str(train_counts[cls]), va='center', fontsize=9)

axes[1].barh(class_names, [test_counts[c] for c in class_names], color=colors)
axes[1].set_xlabel('Number of Images')
axes[1].set_title('Test Set Distribution')
for i, cls in enumerate(class_names):
    axes[1].text(test_counts[cls]+5, i, str(test_counts[cls]), va='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_RESULTS_DIR, 'day3_class_distribution.png'), dpi=120)
plt.show()
print(f'Saved → {DRIVE_RESULTS_DIR}/day3_class_distribution.png')

In [ ]:
# ── Show raw sample images (before transforms) ────────────────
fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(NUM_CLASSES * 2.5, 5))
fig.suptitle('Sample Images Per Class (raw)', fontsize=13)

for col, cls in enumerate(class_names):
    cls_dir = os.path.join(train_root, cls)
    imgs = sorted(os.listdir(cls_dir))[:2]
    for row, fname in enumerate(imgs):
        img = Image.open(os.path.join(cls_dir, fname)).convert('RGB')
        axes[row, col].imshow(img)
        if row == 0:
            axes[row, col].set_title(cls, fontsize=10)
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_RESULTS_DIR, 'day3_raw_samples.png'), dpi=120)
plt.show()

## 4 · Compute Dataset Mean & Std

For transfer learning we'll use **ImageNet stats** (because the backbone was trained on ImageNet), but it's good practice to compute your dataset's own stats for comparison.

In [ ]:
# ── Compute per-channel mean & std ─────────────────────────────
temp_transform = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor()])
temp_dataset   = ImageFolder(train_root, transform=temp_transform)

n_samples = min(2000, len(temp_dataset))
indices   = np.random.choice(len(temp_dataset), n_samples, replace=False)

ch_mean = torch.zeros(3)
ch_std  = torch.zeros(3)

for idx in indices:
    img, _ = temp_dataset[idx]
    for c in range(3):
        ch_mean[c] += img[c].mean()
        ch_std[c]  += img[c].std()

ch_mean /= n_samples
ch_std  /= n_samples

print(f'Dataset mean: ({ch_mean[0]:.4f}, {ch_mean[1]:.4f}, {ch_mean[2]:.4f})')
print(f'Dataset std:  ({ch_std[0]:.4f}, {ch_std[1]:.4f}, {ch_std[2]:.4f})')
print(f'\nImageNet mean: (0.4850, 0.4560, 0.4060)')
print(f'ImageNet std:  (0.2290, 0.2240, 0.2250)')
print(f'\n→ Using ImageNet stats for transfer learning')

del temp_dataset

---
## 5 · Data Pipeline — Transforms & DataLoaders

**Key concepts:**
- `ImageFolder` auto-discovers classes from folder names
- Training transforms include **augmentation** (random crop, flip, color jitter)
- Eval transforms are **deterministic** (center crop only)
- We split training data into **train + validation**

In [ ]:
# ── Educational: Custom Dataset class ─────────────────────────
# This shows what ImageFolder does internally.
# In practice, just use ImageFolder — but know how to write one!

class CustomImageDataset(Dataset):
    """
    Custom Dataset for folder-per-class image data.
    Educational — shows the __len__ and __getitem__ contract.
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        # Discover classes from subdirectory names
        self.classes = sorted([d for d in os.listdir(root_dir)
                               if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        # Build (image_path, label) list
        self.samples = []
        valid_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}
        for cls in self.classes:
            cls_dir = os.path.join(root_dir, cls)
            for fname in sorted(os.listdir(cls_dir)):
                if os.path.splitext(fname)[1].lower() in valid_ext:
                    self.samples.append((os.path.join(cls_dir, fname),
                                        self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

print(f'CustomImageDataset class defined ✅')
print(f'In practice, use ImageFolder — it does the same thing with more features.')

In [ ]:
# ── Transforms ────────────────────────────────────────────────
transform_train = T.Compose([
    T.Resize(int(IMG_SIZE * 1.15)),       # Over-resize for crop margin
    T.RandomCrop(IMG_SIZE),                # Random crop (augmentation)
    T.RandomHorizontalFlip(p=0.5),         # 50% flip (augmentation)
    T.ColorJitter(brightness=0.1, contrast=0.1,
                  saturation=0.1, hue=0.05),  # Colour jitter
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

transform_eval = T.Compose([
    T.Resize(int(IMG_SIZE * 1.15)),
    T.CenterCrop(IMG_SIZE),                # Deterministic crop
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print(f'Training transforms: resize → random crop → flip → jitter → normalize')
print(f'Eval transforms:     resize → center crop → normalize')

In [ ]:
# ── Build datasets & loaders ──────────────────────────────────
# Using ImageFolder (the standard way)
full_train = ImageFolder(train_root, transform=transform_train)
test_set   = ImageFolder(test_root,  transform=transform_eval)

# Train / Validation split
val_size   = int(len(full_train) * VAL_SPLIT)
train_size = len(full_train) - val_size
train_set, val_set = random_split(
    full_train, [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

# DataLoaders
train_loader = DataLoader(train_set, BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'\nDataset: Intel Image Classification')
print(f'Splits — Train: {train_size:,}  Val: {val_size:,}  Test: {len(test_set):,}')
print(f'Classes ({NUM_CLASSES}): {class_names}')
print(f'Batch size: {BATCH_SIZE}  |  Image size: {IMG_SIZE}px')
print(f'Batches — Train: {len(train_loader)}  Val: {len(val_loader)}  Test: {len(test_loader)}')

## 6 · Visualise Sample Batches — Sanity Check!

**Always visualise your data after transforms.** This catches bugs like:
- Wrong normalisation (images look weird)
- Labels don't match images
- Augmentation too aggressive

In [ ]:
def show_batch(loader, title='Sample Batch', n=16):
    """Display a grid of un-normalised images from a DataLoader."""
    imgs, labels = next(iter(loader))

    # Un-normalise for display
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    imgs_disp = (imgs[:n] * std + mean).clamp(0, 1)

    n = min(n, len(imgs_disp))
    cols = min(8, n)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols*2, rows*2.3))
    for i, ax in enumerate(np.array(axes).flat):
        if i < n:
            ax.imshow(imgs_disp[i].permute(1,2,0).numpy())
            ax.set_title(class_names[labels[i]], fontsize=9)
        ax.axis('off')

    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_batch(train_loader, 'Training Batch (augmented)')
show_batch(val_loader,   'Validation Batch (no augment)')

---
## 7 · Model Builder — Same Pattern as Day 2

Reuse the same `build_model` pattern:
1. Load pretrained backbone
2. Replace classifier head for N classes
3. Optionally freeze backbone
4. Re-enable gradients on new head

In [ ]:
SUPPORTED = {
    'resnet18':       (models.resnet18,       ResNet18_Weights.DEFAULT),
    'resnet50':       (models.resnet50,       ResNet50_Weights.DEFAULT),
    'efficientnet_b0':(models.efficientnet_b0, EfficientNet_B0_Weights.DEFAULT),
}

def build_model(backbone_name, num_classes, freeze_backbone):
    builder, weights = SUPPORTED[backbone_name]
    model = builder(weights=weights)

    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad = False

    if 'resnet' in backbone_name:
        in_feat = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_feat, num_classes))
    elif 'efficientnet' in backbone_name:
        in_feat = model.classifier[-1].in_features
        model.classifier[-1] = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_feat, num_classes))

    head = model.fc if 'resnet' in backbone_name else model.classifier[-1]
    for p in head.parameters():
        p.requires_grad = True

    return model.to(DEVICE)


def param_summary(model, label):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'[{label}] Total: {total:,}  Trainable: {trainable:,}  ({100*trainable/total:.1f}%)')


# Preview
print('--- Feature Extraction ---')
m = build_model(BACKBONE, NUM_CLASSES, freeze_backbone=True)
param_summary(m, 'Feature Extraction')
del m

print('\n--- Full Fine-Tuning ---')
m = build_model(BACKBONE, NUM_CLASSES, freeze_backbone=False)
param_summary(m, 'Full Fine-Tuning')
del m

torch.cuda.empty_cache()

## 8 · Training & Evaluation Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        _, pred = out.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()
    return total_loss / total, 100.0 * correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out = model(imgs)
        loss = criterion(out, labels)
        total_loss += loss.item() * imgs.size(0)
        _, pred = out.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return (total_loss / total, 100.0 * correct / total,
            np.array(all_preds), np.array(all_labels))

print('✅ Training functions ready')

In [ ]:
def run_training(backbone_name, num_classes, freeze_backbone, epochs,
                 lr_head, lr_backbone, weight_decay):
    """Full training loop — returns history dict."""
    mode  = 'feature_extraction' if freeze_backbone else 'full_finetune'
    label = 'FROZEN' if freeze_backbone else 'UNFROZEN'

    print(f'\n{"─"*60}')
    print(f'  {backbone_name.upper()} [{label} backbone]')
    print(f'  {epochs} epochs | LR head={lr_head} backbone={lr_backbone}')
    print(f'{"─"*60}')

    model = build_model(backbone_name, num_classes, freeze_backbone)
    param_summary(model, mode)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # Two-LR strategy
    if freeze_backbone:
        params = [p for p in model.parameters() if p.requires_grad]
        optimizer = optim.AdamW(params, lr=lr_head, weight_decay=weight_decay)
    else:
        head = model.fc if 'resnet' in backbone_name else model.classifier[-1]
        head_ids = {id(p) for p in head.parameters()}
        bb_params   = [p for p in model.parameters() if id(p) not in head_ids]
        head_params = list(head.parameters())
        optimizer = optim.AdamW([
            {'params': bb_params,   'lr': lr_backbone},
            {'params': head_params, 'lr': lr_head},
        ], weight_decay=weight_decay)

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
    best_val_acc = 0.0

    t0 = time.time()
    for epoch in range(1, epochs+1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)

        lr_now = optimizer.param_groups[0]['lr']
        print(f'  Ep {epoch:2d}/{epochs}  LR={lr_now:.2e}  '
              f'Train {tr_loss:.3f}/{tr_acc:.1f}%  '
              f'Val {vl_loss:.3f}/{vl_acc:.1f}%')

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(model.state_dict(),
                       os.path.join(DRIVE_MODEL_DIR, f'day3_{backbone_name}_{mode}.pt'))

    elapsed = time.time() - t0
    print(f'\n  Best val acc: {best_val_acc:.1f}%  ({elapsed:.0f}s)')

    # Test evaluation
    model.load_state_dict(torch.load(
        os.path.join(DRIVE_MODEL_DIR, f'day3_{backbone_name}_{mode}.pt'),
        weights_only=True))
    te_loss, te_acc, te_preds, te_labels = evaluate(model, test_loader, criterion)
    print(f'  Test acc: {te_acc:.1f}%')

    # Per-class accuracy
    print(f'\n  Per-class accuracy:')
    for i, cls in enumerate(class_names):
        mask = te_labels == i
        if mask.sum() > 0:
            cls_acc = 100.0 * (te_preds[mask] == i).sum() / mask.sum()
            print(f'    {cls:<15} {cls_acc:.1f}%  ({mask.sum()} samples)')

    history['best_val_acc'] = best_val_acc
    history['test_acc']     = te_acc
    history['mode']         = mode

    del model
    torch.cuda.empty_cache()
    return history

print('✅ run_training ready')

---
## 9 · Strategy A — Feature Extraction (frozen backbone)

In [ ]:
feat_history = run_training(
    backbone_name=BACKBONE,
    num_classes=NUM_CLASSES,
    freeze_backbone=True,
    epochs=EPOCHS,
    lr_head=LR_HEAD,
    lr_backbone=LR_BACKBONE,
    weight_decay=WEIGHT_DECAY,
)

## 10 · Strategy B — Full Fine-Tuning (unfrozen backbone)

In [ ]:
finetune_history = run_training(
    backbone_name=BACKBONE,
    num_classes=NUM_CLASSES,
    freeze_backbone=False,
    epochs=EPOCHS,
    lr_head=LR_HEAD,
    lr_backbone=LR_BACKBONE,
    weight_decay=WEIGHT_DECAY,
)

---
## 11 · Comparison — Feature Extraction vs Fine-Tuning

In [ ]:
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Day 3 — {BACKBONE.upper()} on Intel Image Classification', fontsize=14)

colors = {'fe_tr': '#4C72B0', 'fe_vl': '#55A868',
          'ft_tr': '#C44E52', 'ft_vl': '#DD8452'}

for i, (metric, ylabel) in enumerate([('acc','Accuracy (%)'), ('loss','Loss')]):
    ax = axes[i]
    ax.plot(epochs_range, feat_history[f'train_{metric}'],
            color=colors['fe_tr'], ls='--', label='FeatExt Train')
    ax.plot(epochs_range, feat_history[f'val_{metric}'],
            color=colors['fe_vl'], label='FeatExt Val')
    ax.plot(epochs_range, finetune_history[f'train_{metric}'],
            color=colors['ft_tr'], ls='--', label='FineTune Train')
    ax.plot(epochs_range, finetune_history[f'val_{metric}'],
            color=colors['ft_vl'], label='FineTune Val')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.set_title(f'Training {ylabel}')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_RESULTS_DIR, f'day3_{BACKBONE}_comparison.png'), dpi=120)
plt.show()
print(f'Saved → {DRIVE_RESULTS_DIR}/day3_{BACKBONE}_comparison.png')

In [ ]:
# ── Results Summary ───────────────────────────────────────────
print('\n' + '=' * 55)
print('RESULTS SUMMARY — CUSTOM DATASET')
print('=' * 55)
print(f'{"Strategy":<25} {"Val Acc":>10} {"Test Acc":>10}')
print('-' * 47)
print(f'{"Feature Extraction":<25} {feat_history["best_val_acc"]:>9.1f}%'
      f' {feat_history["test_acc"]:>9.1f}%')
print(f'{"Full Fine-Tuning":<25} {finetune_history["best_val_acc"]:>9.1f}%'
      f' {finetune_history["test_acc"]:>9.1f}%')

winner = ('Full Fine-Tuning' if finetune_history['test_acc'] > feat_history['test_acc']
          else 'Feature Extraction')
diff = abs(finetune_history['test_acc'] - feat_history['test_acc'])
print(f'\n✅ Winner: {winner}  (by {diff:.1f}%)')

# Save results JSON
results = {
    'dataset': 'Intel Image Classification',
    'backbone': BACKBONE,
    'epochs': EPOCHS,
    'num_classes': NUM_CLASSES,
    'class_names': class_names,
    'feature_extraction': {
        'best_val_acc': feat_history['best_val_acc'],
        'test_acc': feat_history['test_acc'],
    },
    'full_finetune': {
        'best_val_acc': finetune_history['best_val_acc'],
        'test_acc': finetune_history['test_acc'],
    },
}
results_path = os.path.join(DRIVE_RESULTS_DIR, f'day3_{BACKBONE}_custom_results.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nResults JSON → {results_path}')

---
## 12 · Key Takeaways

**Day 3 taught you:**

1. **`ImageFolder`** auto-loads folder-per-class datasets — the easiest path
2. **Custom `Dataset`** gives full flexibility (CSV labels, multi-label, etc.)
3. **Training transforms ≠ Eval transforms** — augmentation only during training!
4. **Always sanity-check** your data pipeline: visualise batches, check class counts
5. **ImageNet normalisation** is required when using pretrained backbones
6. **The same transfer learning patterns from Day 2** work on any custom dataset

**Next → Day 4:** Data augmentation ablation with **Albumentations**
- Establish a no-augmentation baseline
- Add Albumentations transforms
- Compare: does augmentation help on this dataset?